# Structured output at scale: retries, validation, and fallback routing

A data extraction pipeline parses 10K invoices per day from PDF inputs into a strict Pydantic schema. The team's first pass uses OpenAI structured output with strict mode. The schema rejects 8% of responses because vendor names span multiple lines, currency fields go missing, and line items scramble across page breaks. You have 75 minutes to build a robust three-layer pipeline against 500 labelled invoice PDFs supplied by the lab: strict Pydantic validation, an automatic-retry loop on schema failure that feeds the validator's error message back into the prompt, and a fallback to Claude Sonnet 4.6 on persistent failure. Target 99% success across the 500-example test set.

The killer moment is the trace. After 500 invoices, you have a CSV that shows exactly where retries fired, where fallbacks fired, and which combination of LLM hops produced each successful extraction. That artifact is the deployment shape for every LLM-into-database pipeline in 2026.

Estimated time: 75 minutes. Cost: about $1.50 to $2.00 on a clean run; up to $3.00 if you re-run a task while debugging. GPT-4o-mini does the bulk of the work at $0.15 / $0.60 per million tokens; Claude Sonnet 4.6 steps in on the failures at $3 / $15.

Two bucks. GPT-4o-mini does the bulk; Claude Sonnet steps in only when OpenAI's strict mode rejects after retries.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. labrun-checks==0.7.0 matches the rest of the AI/ML track.
# pymupdf is the PDF text extractor; reportlab is the deterministic PDF generator
# for the 500-invoice corpus. Strict version pins so the synthetic corpus is
# byte-stable across sessions.

!pip install --quiet labrun-checks==0.7.0 openai==1.54.4 anthropic==0.42.0 boto3==1.35.71 pydantic==2.10.2 pymupdf==1.25.1 reportlab==4.2.5

In [ ]:
# Imports and per-lab constants. Constants are declared up front so every
# downstream cell references the same bucket name, model names, and file paths.

import atexit
import csv
import getpass
import io
import json
import os
import random
import sys
import time
from typing import Optional

import boto3
from botocore.exceptions import ClientError
from pydantic import BaseModel, Field, ValidationError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "structured-output-with-retries-and-fallback"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID

REGION = "us-east-1"
BUCKET_NAME = "labrun-structured-output-with-retries-and-fallback-bucket"

# Local artifacts. Colab mounts /content; everything we write lives under it.
INVOICES_DIR = "/content/invoices"
RESULTS_CSV = "/content/extraction_results.csv"

# Model choices. GPT-4o-mini is the primary structured-output workhorse;
# Claude Sonnet 4.6 is the fallback when OpenAI strict mode keeps rejecting
# the same invoice. Both names are pinned so the lab is reproducible.
OPENAI_MODEL = "gpt-4o-mini"
ANTHROPIC_FALLBACK_MODEL = "claude-sonnet-4-6"

# Corpus parameters. 500 PDFs total, roughly 8% deliberately malformed so the
# retry loop and the fallback both get exercised on a single end-to-end run.
TOTAL_INVOICES = 500
MALFORMED_RATIO = 0.08
RANDOM_SEED = 17  # deterministic so the malformed set is the same every time

In [ ]:
# NBVAL_SKIP
# BYOK setup: session token, OPENAI_API_KEY, ANTHROPIC_API_KEY,
# AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY. We validate every credential
# with the cheapest possible call before registering the session so a
# wrong key shows up here, not 200 PDFs into the batch.

import openai
import anthropic

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
OPENAI_API_KEY = getpass.getpass("OPENAI_API_KEY: ").strip()
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip()

if not all([OPENAI_API_KEY, ANTHROPIC_API_KEY, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY]):
    print("All four credentials are required. Re-run this cell.")
    raise SystemExit(1)

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION

# AWS STS GetCallerIdentity. Surfaces account id, fails fast if the key is
# revoked or pointing at the wrong account.
try:
    sts = boto3.client("sts", region_name=REGION)
    identity = sts.get_caller_identity()
    print(f"AWS credentials validated. Account: {identity['Account']}. Region: {REGION}.")
except ClientError as exc:
    print(f"AWS credentials rejected: {exc!r}")
    print("Refresh your AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and re-run this cell.")
    raise SystemExit(1)

# Cheap OpenAI ping. One gpt-4o-mini call with a 5-token cap: under $0.001.
try:
    openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
    _ping = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        max_tokens=5,
        messages=[{"role": "user", "content": "reply with the single word: ok"}],
    )
    print(f"OpenAI credentials validated. {OPENAI_MODEL} replied: {_ping.choices[0].message.content!r}")
except Exception as exc:
    print(f"OpenAI key rejected: {exc!r}")
    print("Refresh OPENAI_API_KEY and re-run this cell.")
    raise SystemExit(1)

# Cheap Anthropic ping. One Sonnet call with an 8-token cap: a fraction of a cent.
try:
    anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    _ping = anthropic_client.messages.create(
        model=ANTHROPIC_FALLBACK_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with the single word: ok"}],
    )
    print(f"Anthropic credentials validated. {ANTHROPIC_FALLBACK_MODEL} replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    print("Refresh ANTHROPIC_API_KEY and re-run this cell.")
    raise SystemExit(1)

s3 = boto3.client("s3", region_name=REGION)

AWS_CREDENTIALS = {
    "aws_access_key_id": AWS_ACCESS_KEY_ID,
    "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
    "region": REGION,
}

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, custom Lab07CleanupAdapter, atexit safety net, orphan scan.
# The manifest is module-level so atexit and the cleanup cell both see the
# same list. Reverse-creation order: S3 bucket (with all objects) first,
# then local files.
# TODO: migrate this adapter into labrun_checks.adapters.aws once the
# package supports mixed cloud-plus-local manifests as a first-class case.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws s3 rb s3://{BUCKET_NAME} --force --region {REGION}"
        ),
    ),
    CleanupResource(
        type="local_file",
        id=RESULTS_CSV,
        region="local",
        cli_delete_command=f"rm -f {RESULTS_CSV}",
    ),
    CleanupResource(
        type="local_dir",
        id=INVOICES_DIR,
        region="local",
        cli_delete_command=f"rm -rf {INVOICES_DIR}",
    ),
]


class Lab07CleanupAdapter:
    """S3-plus-local adapter for Lab 07. Empties and deletes the lab bucket;
    removes the local results CSV and the invoices directory.

    TODO: fold into labrun_checks.adapters.aws once the package grows a
    first-class "local file" resource type. For now the adapter is inline
    so the lab is self-contained and the AWS adapter stays cloud-only.
    """

    def __init__(self, s3_client, region):
        self._s3 = s3_client
        self._region = region

    def _empty_bucket(self, bucket):
        paginator = self._s3.get_paginator("list_objects_v2")
        to_delete = []
        for page in paginator.paginate(Bucket=bucket):
            for obj in page.get("Contents", []):
                to_delete.append({"Key": obj["Key"]})
                if len(to_delete) == 1000:
                    self._s3.delete_objects(Bucket=bucket, Delete={"Objects": to_delete})
                    to_delete = []
        if to_delete:
            self._s3.delete_objects(Bucket=bucket, Delete={"Objects": to_delete})

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "s3_bucket":
            try:
                self._empty_bucket(rid)
                self._s3.delete_bucket(Bucket=rid)
            except ClientError as exc:
                code_str = exc.response.get("Error", {}).get("Code", "")
                if code_str in ("NoSuchBucket", "NotFound"):
                    return
                raise
        elif rtype == "local_file":
            try:
                os.remove(rid)
            except FileNotFoundError:
                pass
        elif rtype == "local_dir":
            if not os.path.exists(rid):
                return
            for root, dirs, files in os.walk(rid, topdown=False):
                for name in files:
                    try:
                        os.remove(os.path.join(root, name))
                    except FileNotFoundError:
                        pass
                for name in dirs:
                    try:
                        os.rmdir(os.path.join(root, name))
                    except FileNotFoundError:
                        pass
            try:
                os.rmdir(rid)
            except FileNotFoundError:
                pass
        else:
            raise ValueError(f"Lab07CleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "s3_bucket":
            try:
                self._s3.head_bucket(Bucket=rid)
                return True
            except ClientError as exc:
                code_str = exc.response.get("Error", {}).get("Code", "")
                if code_str in ("404", "NoSuchBucket", "NotFound"):
                    return False
                return True
        if rtype == "local_file":
            return os.path.exists(rid)
        if rtype == "local_dir":
            return os.path.isdir(rid)
        return False

    def tag_scan(self, credentials, lab_slug, region):
        # Tag-based orphan scan for this lab is bucket-only. The lab tags
        # its bucket on creation with labrun:lab-slug; the Resource Groups
        # Tagging API surfaces it. Local files are not in scope (they live
        # in Colab's ephemeral /content and disappear with the runtime).
        try:
            rgt = boto3.client(
                "resourcegroupstaggingapi",
                region_name=region,
                aws_access_key_id=credentials.get("aws_access_key_id"),
                aws_secret_access_key=credentials.get("aws_secret_access_key"),
            )
            resp = rgt.get_resources(
                TagFilters=[{"Key": LAB_TAG_KEY, "Values": [lab_slug]}],
            )
            return [r["ResourceARN"] for r in resp.get("ResourceTagMappingList", [])]
        except Exception:
            return []


CLEANUP_ADAPTER = Lab07CleanupAdapter(s3_client=s3, region=REGION)


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    arns = CLEANUP_ADAPTER.tag_scan(AWS_CREDENTIALS, LAB_TAG_VALUE, REGION)
    # Also surface a leftover bucket even before tags propagate (the tagging
    # API is eventually consistent; head_bucket is immediate).
    try:
        s3.head_bucket(Bucket=BUCKET_NAME)
        arns.append(f"arn:aws:s3:::{BUCKET_NAME}")
    except ClientError:
        pass
    return list(set(arns))


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged for lab {LAB_ID!r} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Generate the 500-invoice corpus and upload to S3

Five hundred PDFs, deterministic from `RANDOM_SEED=17` so the malformed subset is identical across sessions. Roughly 8% are deliberately broken in one of three ways: line items scrambled across columns, currency field missing entirely, or vendor names that wrap across two lines so the naive single-line text extractor drops them. The rest are clean.

Each PDF has a ground-truth label (clean or malformed) written into a sidecar JSON; the eval task in Checkpoint 4 reads the labels back to compute the success rate.

The bucket is created with the `labrun:lab-slug` tag so the orphan scan can find it on the next session if cleanup fails.

In [ ]:
# NBVAL_SKIP
# Synthetic invoice PDF generator. Deterministic given RANDOM_SEED. Writes
# 500 PDFs under INVOICES_DIR, a labels.json sidecar, then uploads every PDF
# to s3://BUCKET_NAME/invoices/.

from reportlab.lib.pagesizes import LETTER
from reportlab.pdfgen import canvas

os.makedirs(INVOICES_DIR, exist_ok=True)

VENDORS_CLEAN = [
    "Acme Corp", "Globex Industries", "Initech LLC", "Umbrella Co",
    "Wayne Enterprises", "Stark Industries", "Wonka Goods", "Pied Piper",
    "Hooli Systems", "Tyrell Robotics",
]
# Multi-line vendor names: the trap is that a naive single-page text-extract
# returns these as two separate strings. PyMuPDF preserves layout enough that
# the schema-aware retry can recover; pdfplumber sometimes drops the second line.
VENDORS_MULTILINE = [
    "Northeastern Regional\nManufacturing Group",
    "Pacific Coastal Logistics\nand Freight Holdings",
    "Continental Wholesale\nDistribution Partners",
]
CURRENCIES = ["USD", "EUR", "GBP", "JPY"]
ITEM_DESCRIPTIONS = [
    "Widget assembly", "Service hours", "Maintenance fee",
    "Software license", "Shipping", "Materials", "Consulting",
    "Training session", "Equipment rental", "Inspection",
]


def _build_clean_pdf(path, invoice_no, vendor, currency, items, total):
    c = canvas.Canvas(path, pagesize=LETTER)
    c.setFont("Helvetica", 12)
    c.drawString(72, 740, f"Invoice Number: {invoice_no}")
    c.drawString(72, 720, f"Vendor: {vendor}")
    c.drawString(72, 700, f"Currency: {currency}")
    y = 660
    c.drawString(72, y, "Line Items:")
    y -= 20
    for desc, amount in items:
        c.drawString(90, y, f"- {desc}: {amount:.2f}")
        y -= 18
    c.drawString(72, y - 10, f"Total: {total:.2f}")
    c.save()


def _build_multiline_vendor_pdf(path, invoice_no, vendor, currency, items, total):
    """Vendor name wraps across two lines, no explicit 'Vendor:' label on line 2."""
    c = canvas.Canvas(path, pagesize=LETTER)
    c.setFont("Helvetica", 12)
    c.drawString(72, 740, f"Invoice Number: {invoice_no}")
    line1, line2 = vendor.split("\n", 1)
    c.drawString(72, 720, f"Vendor: {line1}")
    c.drawString(72, 705, line2)  # second line drops below; no explicit label
    c.drawString(72, 685, f"Currency: {currency}")
    y = 655
    c.drawString(72, y, "Line Items:")
    y -= 20
    for desc, amount in items:
        c.drawString(90, y, f"- {desc}: {amount:.2f}")
        y -= 18
    c.drawString(72, y - 10, f"Total: {total:.2f}")
    c.save()


def _build_missing_currency_pdf(path, invoice_no, vendor, items, total):
    """Currency field omitted entirely. The strict schema must reject the
    first OpenAI response that hallucinates a default."""
    c = canvas.Canvas(path, pagesize=LETTER)
    c.setFont("Helvetica", 12)
    c.drawString(72, 740, f"Invoice Number: {invoice_no}")
    c.drawString(72, 720, f"Vendor: {vendor}")
    # No currency line at all.
    y = 690
    c.drawString(72, y, "Line Items:")
    y -= 20
    for desc, amount in items:
        c.drawString(90, y, f"- {desc}: {amount:.2f}")
        y -= 18
    c.drawString(72, y - 10, f"Total: {total:.2f}")
    c.save()


def _build_scrambled_items_pdf(path, invoice_no, vendor, currency, items, total):
    """Line items split across two columns, descriptions on the left and
    amounts on the right under a separate header. Naive extractors zip the
    two columns wrong and the schema rejects."""
    c = canvas.Canvas(path, pagesize=LETTER)
    c.setFont("Helvetica", 12)
    c.drawString(72, 740, f"Invoice Number: {invoice_no}")
    c.drawString(72, 720, f"Vendor: {vendor}")
    c.drawString(72, 700, f"Currency: {currency}")
    c.drawString(72, 670, "Descriptions:")
    c.drawString(300, 670, "Amounts:")
    y = 650
    for desc, amount in items:
        c.drawString(90, y, f"- {desc}")
        c.drawString(310, y, f"{amount:.2f}")
        y -= 18
    c.drawString(72, y - 10, f"Total: {total:.2f}")
    c.save()


def generate_corpus(out_dir, n, malformed_ratio, seed):
    rng = random.Random(seed)
    labels = {}
    malformed_count = int(round(n * malformed_ratio))
    malformed_indices = set(rng.sample(range(n), malformed_count))
    malformed_kinds = ["multiline_vendor", "missing_currency", "scrambled_items"]
    for i in range(n):
        invoice_no = f"INV-{10000 + i:05d}"
        currency = rng.choice(CURRENCIES)
        n_items = rng.randint(2, 5)
        items = [
            (rng.choice(ITEM_DESCRIPTIONS), round(rng.uniform(10.0, 990.0), 2))
            for _ in range(n_items)
        ]
        total = round(sum(a for _, a in items), 2)
        path = os.path.join(out_dir, f"{invoice_no}.pdf")
        if i in malformed_indices:
            kind = malformed_kinds[i % len(malformed_kinds)]
            if kind == "multiline_vendor":
                vendor = rng.choice(VENDORS_MULTILINE)
                _build_multiline_vendor_pdf(path, invoice_no, vendor, currency, items, total)
            elif kind == "missing_currency":
                vendor = rng.choice(VENDORS_CLEAN)
                _build_missing_currency_pdf(path, invoice_no, vendor, items, total)
            else:
                vendor = rng.choice(VENDORS_CLEAN)
                _build_scrambled_items_pdf(path, invoice_no, vendor, currency, items, total)
            labels[invoice_no] = {"malformed": True, "kind": kind}
        else:
            vendor = rng.choice(VENDORS_CLEAN)
            _build_clean_pdf(path, invoice_no, vendor, currency, items, total)
            labels[invoice_no] = {"malformed": False, "kind": "clean"}
    with open(os.path.join(out_dir, "labels.json"), "w") as f:
        json.dump(labels, f, indent=2)
    return labels


print(f"Generating {TOTAL_INVOICES} invoice PDFs (deterministic seed={RANDOM_SEED})...")
LABELS = generate_corpus(INVOICES_DIR, TOTAL_INVOICES, MALFORMED_RATIO, RANDOM_SEED)
print(f"Wrote {TOTAL_INVOICES} PDFs to {INVOICES_DIR}. Malformed count: "
      f"{sum(1 for v in LABELS.values() if v['malformed'])} ({MALFORMED_RATIO * 100:.0f}%).")

# Create the S3 bucket. us-east-1 must NOT pass LocationConstraint; every
# other region must. Tag the bucket immediately so the orphan scan finds it.
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(
        Bucket=BUCKET_NAME,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    print(f"Created S3 bucket: {BUCKET_NAME}")
except ClientError as exc:
    code_str = exc.response.get("Error", {}).get("Code", "")
    if code_str in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket {BUCKET_NAME} already exists in this account; reusing.")
    else:
        raise

print(f"Uploading {TOTAL_INVOICES} PDFs to s3://{BUCKET_NAME}/invoices/ ...")
for invoice_no in LABELS.keys():
    local_path = os.path.join(INVOICES_DIR, f"{invoice_no}.pdf")
    s3.upload_file(local_path, BUCKET_NAME, f"invoices/{invoice_no}.pdf")
print("Upload complete.")

## Task 1: Define the strict Pydantic schema

The schema is the single source of truth for what a successful invoice extraction looks like. OpenAI's structured-output strict mode binds to it; the retry loop reads its `ValidationError` text; the Claude fallback uses the same shape as a tool input schema. If the schema is wrong, every layer downstream is wrong.

Required fields (all top-level required, no Optional unless the data is genuinely optional):

- `invoice_number: str`
- `vendor_name: str`
- `line_items: list[LineItem]` where each `LineItem` is `{ description: str, amount: float }`. At least one item is required.
- `total_amount: float`
- `currency: str` (3-letter ISO code)

The checkpoint runs your schema against two payloads:

1. A known-good payload that should validate clean.
2. A known-bad payload (missing `currency`) that should raise `ValidationError`.

In [ ]:
# Task 1: define the strict Pydantic schema for an invoice. Level 2 skeleton.
# The shape of LineItem and Invoice is yours to write; the test payloads at
# the bottom are fixed so the checkpoint reads the same names.


# YOUR CODE: define LineItem(BaseModel) with two required fields:
# YOUR CODE:   description: str
# YOUR CODE:   amount: float


# YOUR CODE: define Invoice(BaseModel) with five required fields:
# YOUR CODE:   invoice_number: str
# YOUR CODE:   vendor_name: str
# YOUR CODE:   line_items: list[LineItem]    (at least one item)
# YOUR CODE:   total_amount: float
# YOUR CODE:   currency: str                 (3-letter ISO code)


# Fixed test payloads. The known-bad one is missing the `currency` field.
KNOWN_GOOD_PAYLOAD = {
    "invoice_number": "INV-99999",
    "vendor_name": "Acme Corp",
    "line_items": [{"description": "Widget assembly", "amount": 100.0}],
    "total_amount": 100.0,
    "currency": "USD",
}

KNOWN_BAD_PAYLOAD = {
    "invoice_number": "INV-99998",
    "vendor_name": "Acme Corp",
    "line_items": [{"description": "Widget assembly", "amount": 100.0}],
    "total_amount": 100.0,
    # currency intentionally omitted
}

# Quick local self-check so the student sees the result before the checkpoint.
try:
    good = Invoice.model_validate(KNOWN_GOOD_PAYLOAD)
    print(f"Known-good validates clean. invoice_number={good.invoice_number}")
except NameError:
    print("Invoice schema is not defined yet; fill in the skeleton above.")
except ValidationError as exc:
    print(f"Known-good unexpectedly failed validation: {exc.errors()}")

try:
    Invoice.model_validate(KNOWN_BAD_PAYLOAD)
    print("Known-bad UNEXPECTEDLY validated; the schema is too loose.")
except NameError:
    pass
except ValidationError as exc:
    print(f"Known-bad correctly rejected. Field that failed: "
          f"{[e['loc'] for e in exc.errors()]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: the schema accepts the known-good payload and rejects the
# known-bad payload (the one missing currency). We inspect the schema class
# itself; we do not call OpenAI.


def checkpoint_1(session):
    try:
        schema_class = Invoice
    except NameError:
        return CheckpointResult(
            status="fail",
            error_reason="`Invoice` is not defined. Fill in the schema skeleton in the task cell above.",
        )

    # Known-good must validate.
    try:
        schema_class.model_validate(KNOWN_GOOD_PAYLOAD)
    except ValidationError as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Schema rejected the known-good payload. Errors: {exc.errors()}. "
                "Re-check field names and types against the required list."
            ),
        )

    # Known-bad (missing currency) must raise ValidationError.
    try:
        schema_class.model_validate(KNOWN_BAD_PAYLOAD)
    except ValidationError:
        pass
    else:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "Schema accepted the known-bad payload (missing currency). "
                "Strict mode requires every top-level field; mark currency as required, "
                "not Optional."
            ),
        )

    # Sanity-check the field set so a too-permissive schema does not slip
    # through (for example, one that declared all fields Optional[...] = None).
    required_fields = {"invoice_number", "vendor_name", "line_items", "total_amount", "currency"}
    declared = set(schema_class.model_fields.keys())
    missing = required_fields - declared
    if missing:
        return CheckpointResult(
            status="fail",
            error_reason=f"Schema is missing required fields: {sorted(missing)}.",
        )

    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The known-bad payload omits `currency`. If your schema does not reject it, your `currency` field is either Optional or has a default value. Pydantic v2 fields are required by default only when they have no default and no `Optional[...]` annotation.

</details>

<details><summary>Hint 2 (direction)</summary>

Two Pydantic classes. The inner one (`LineItem`) is two fields. The outer one (`Invoice`) declares `line_items: list[LineItem]` and the rest. No `Optional`, no `= None`, no `Field(default=...)` on any of the required fields. For at-least-one-item, `Field(..., min_length=1)` on `line_items` is the idiomatic guard.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working schema:

```python
class LineItem(BaseModel):
    description: str
    amount: float


class Invoice(BaseModel):
    invoice_number: str
    vendor_name: str
    line_items: list[LineItem] = Field(..., min_length=1)
    total_amount: float
    currency: str
```

</details>

**Wallet check.** Schema definition is free. The two cred-validation calls in setup cost a fraction of a cent each. Bucket + 500 PDF uploads is in the noise of the S3 free tier. Spent so far: under one cent. Your morning coffee cost two hundred times more.

## Task 2: Retry loop with the validator error fed back into the prompt

The structured-output call against GPT-4o-mini rejects on roughly 8% of the corpus in strict mode. The retry that does NOT include the validator's error text in the next prompt typically lifts success to about 95%. The retry that DOES include the `ValidationError.errors()` text typically lifts it to over 99%. The whole win is "tell the model what was wrong, not just that something was wrong."

Build a function `extract_with_retry(pdf_text, max_retries=2)` that:

1. Calls `openai_client.beta.chat.completions.parse(model=..., messages=..., response_format=Invoice)`.
2. If parsing succeeds, returns `(Invoice instance, trace_dict)`.
3. If `ValidationError` is raised (or the model returns a refusal), constructs a retry message that includes the error text and re-calls the model.
4. Returns `(None, trace_dict)` after `max_retries` consecutive failures so Task 3's fallback layer can take over.

The trace dict must include a `retries` list where each entry is the error string fed into the retry prompt. Checkpoint 2 reads that list to confirm the retry prompt carried the error context.

In [ ]:
# NBVAL_SKIP
# Task 2: build the retry helper. The known-malformed test PDF below
# is a multi-line-vendor case; it usually trips GPT-4o-mini on the first
# attempt and recovers on the retry with error feedback.

EXTRACTION_SYSTEM_PROMPT = (
    "You are an invoice-extraction service. Given the raw text of an invoice PDF, "
    "extract the structured fields. Every field is required. The currency must be "
    "a 3-letter ISO code. Line items are a list with description and amount."
)


import fitz  # pymupdf


def extract_pdf_text(path):
    """Pull all text from a PDF, preserving line order. PyMuPDF holds multi-line
    vendor names together better than pdfplumber's default extractor."""
    doc = fitz.open(path)
    parts = []
    for page in doc:
        parts.append(page.get_text("text"))
    doc.close()
    return "\n".join(parts)


def extract_with_retry(pdf_text, max_retries=2):
    """Call GPT-4o-mini with strict structured-output. On ValidationError,
    re-prompt with the error text. Return (Invoice or None, trace_dict).

    The trace dict shape:
        {"retries": [str, ...],   # one entry per retry attempt; the error text
         "attempts": int,         # how many total OpenAI calls
         "openai_status": "ok" | "exhausted"}
    """
    trace = {"retries": [], "attempts": 0, "openai_status": "exhausted"}

    user_msg = (
        "Extract the structured fields from this invoice text. "
        "Return JSON matching the Invoice schema.\n\n"
        f"INVOICE TEXT:\n{pdf_text}"
    )

    # YOUR CODE: loop up to (max_retries + 1) times. On each iteration:
    # YOUR CODE:   1. trace["attempts"] += 1
    # YOUR CODE:   2. call openai_client.beta.chat.completions.parse(
    # YOUR CODE:        model=OPENAI_MODEL,
    # YOUR CODE:        messages=[
    # YOUR CODE:          {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
    # YOUR CODE:          {"role": "user",   "content": user_msg},
    # YOUR CODE:        ],
    # YOUR CODE:        response_format=Invoice,
    # YOUR CODE:      )
    # YOUR CODE:   3. if the parsed instance is present AND not a refusal:
    # YOUR CODE:        trace["openai_status"] = "ok"
    # YOUR CODE:        return (parsed, trace)
    # YOUR CODE:   4. on ValidationError or refusal: capture err_text =
    # YOUR CODE:        json.dumps(exc.errors()) (or the refusal string).
    # YOUR CODE:        trace["retries"].append(err_text).
    # YOUR CODE:        Re-build user_msg to include both the original text
    # YOUR CODE:        AND a "PRIOR ATTEMPT FAILED VALIDATION WITH: ..." block.
    # YOUR CODE:        Continue the loop.
    # YOUR CODE: After the loop, return (None, trace).

    return (None, trace)


# Find one known-malformed PDF to use as the test. The first multiline-vendor
# entry in LABELS is the most reliable trigger for strict-mode rejection.
_TEST_INVOICE_NO = next(
    invno for invno, info in LABELS.items()
    if info.get("malformed") and info.get("kind") == "multiline_vendor"
)
_TEST_PDF_PATH = os.path.join(INVOICES_DIR, f"{_TEST_INVOICE_NO}.pdf")
_TEST_TEXT = extract_pdf_text(_TEST_PDF_PATH)

print(f"Running retry test on a known-malformed PDF: {_TEST_INVOICE_NO}")
RETRY_TEST_INSTANCE, RETRY_TEST_TRACE = extract_with_retry(_TEST_TEXT, max_retries=2)
print(f"Attempts: {RETRY_TEST_TRACE['attempts']}")
print(f"Retries fed back: {len(RETRY_TEST_TRACE['retries'])}")
print(f"OpenAI status: {RETRY_TEST_TRACE['openai_status']}")
if RETRY_TEST_INSTANCE is not None:
    print(f"Extracted vendor: {RETRY_TEST_INSTANCE.vendor_name!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: the retry helper fired at least once AND the retry prompt
# carried the ValidationError text from the failed attempt. We do not require
# eventual success here; the fallback layer in Task 3 takes care of the cases
# where GPT-4o-mini cannot recover.


def checkpoint_2(session):
    try:
        trace = RETRY_TEST_TRACE
    except NameError:
        return CheckpointResult(
            status="fail",
            error_reason="RETRY_TEST_TRACE is not defined; Task 2 cell did not run.",
        )

    if not isinstance(trace, dict):
        return CheckpointResult(
            status="fail",
            error_reason=f"Expected trace dict, got {type(trace).__name__}.",
        )

    if trace.get("attempts", 0) < 2:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Only {trace.get('attempts', 0)} OpenAI call(s) recorded. "
                "The retry loop should have made at least 2 attempts on the "
                "multi-line-vendor PDF. Confirm the loop iterates on ValidationError."
            ),
        )

    retries = trace.get("retries", [])
    if not isinstance(retries, list) or not retries:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "trace['retries'] is empty. The retry helper must append the "
                "validator's error text to trace['retries'] each time it re-prompts."
            ),
        )

    # The retry payload must look like the Pydantic .errors() output: a list of
    # dicts with 'loc' / 'msg' / 'type'. Accept JSON-stringified or repr.
    sample = retries[0]
    looks_like_pydantic = (
        "loc" in sample and ("msg" in sample or "type" in sample)
    )
    if not looks_like_pydantic:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "The retry prompt does not include Pydantic's ValidationError.errors() "
                "output. Pass `json.dumps(exc.errors())` (or repr(exc.errors())) into "
                "the retry message so the model sees what failed."
            ),
        )

    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two failure modes for this checkpoint. Either the retry never fires (the loop runs once and returns), or the retry fires but the second prompt is identical to the first. The model needs the error text to know what to fix.

</details>

<details><summary>Hint 2 (direction)</summary>

Wrap `openai_client.beta.chat.completions.parse(...)` in `try / except ValidationError`. On the except branch, capture `err = exc.errors()`, append `json.dumps(err)` to `trace["retries"]`, and rebuild `user_msg` so the next iteration sees a block like:

```
PRIOR ATTEMPT FAILED VALIDATION WITH:
[{"loc": ["currency"], "msg": "Field required", "type": "missing"}]
Fix the issues and return valid JSON.
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working `extract_with_retry`:

```python
def extract_with_retry(pdf_text, max_retries=2):
    trace = {"retries": [], "attempts": 0, "openai_status": "exhausted"}
    base_user_msg = (
        "Extract the structured fields from this invoice text. "
        "Return JSON matching the Invoice schema.\n\n"
        f"INVOICE TEXT:\n{pdf_text}"
    )
    user_msg = base_user_msg
    for _ in range(max_retries + 1):
        trace["attempts"] += 1
        try:
            completion = openai_client.beta.chat.completions.parse(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                response_format=Invoice,
            )
            choice = completion.choices[0].message
            if getattr(choice, "refusal", None):
                err_text = f"refusal: {choice.refusal}"
                trace["retries"].append(err_text)
                user_msg = (
                    base_user_msg
                    + f"\n\nPRIOR ATTEMPT FAILED VALIDATION WITH:\n{err_text}\n"
                      "Fix the issues and return valid JSON."
                )
                continue
            parsed = choice.parsed
            if parsed is None:
                trace["retries"].append("parsed=None")
                continue
            trace["openai_status"] = "ok"
            return (parsed, trace)
        except ValidationError as exc:
            err_text = json.dumps(exc.errors())
            trace["retries"].append(err_text)
            user_msg = (
                base_user_msg
                + f"\n\nPRIOR ATTEMPT FAILED VALIDATION WITH:\n{err_text}\n"
                  "Fix the issues and return valid JSON."
            )
    return (None, trace)
```

</details>

**Wallet check.** One PDF, two or three GPT-4o-mini calls under 1K tokens each. Spent on this task: about $0.01. Cumulative session: roughly $0.02. Still well under the price of a coffee.

## Task 3: Fallback to Claude Sonnet on 3 consecutive retry failures

Even with the error-feedback retry, a small slice of invoices breaks GPT-4o-mini's strict mode every time. Three consecutive failures on the same invoice is the production heuristic for "this model cannot do this one; route to the next model."

The fallback is Claude Sonnet 4.6 with tool-use enforcement. The same Pydantic schema is converted to a JSON Schema and registered as the input schema of a single tool called `submit_invoice`. We force the model to call exactly that tool via `tool_choice={"type": "tool", "name": "submit_invoice"}`. The result is the same shape as the OpenAI parse output, with effectively zero schema drift.

Build `extract_with_fallback(pdf_text)` that:

1. Calls `extract_with_retry(...)` with `max_retries=2` (so a total of 3 OpenAI attempts).
2. If OpenAI status is `ok`, returns `(instance, trace)` with `trace["used_fallback"] = False`.
3. If OpenAI exhausted, calls Claude Sonnet with the tool-use schema, validates the tool input against the Pydantic schema, and returns `(instance, trace)` with `trace["used_fallback"] = True`. Also stores `trace["fallback_model"] = ANTHROPIC_FALLBACK_MODEL`.

The checkpoint runs against a known-3x-failure PDF (a scrambled-items case that GPT-4o-mini reliably mis-parses).

In [ ]:
# NBVAL_SKIP
# Task 3: build the fallback wrapper.

FALLBACK_TOOL = {
    "name": "submit_invoice",
    "description": "Submit the structured invoice extraction matching the strict schema.",
    "input_schema": Invoice.model_json_schema(),
}

FALLBACK_SYSTEM = (
    "You are an invoice-extraction service. The OpenAI primary failed three times. "
    "Read the invoice text, then call the submit_invoice tool with every field filled in. "
    "Currency must be a 3-letter ISO code. line_items requires at least one entry."
)


def extract_with_fallback(pdf_text):
    """Run extract_with_retry. If GPT-4o-mini exhausts, fall back to Claude Sonnet.

    Returns (instance_or_None, trace) where trace carries:
        retries, attempts, openai_status, used_fallback, fallback_model
    """
    instance, trace = extract_with_retry(pdf_text, max_retries=2)
    trace["used_fallback"] = False
    trace["fallback_model"] = None

    if instance is not None:
        return (instance, trace)

    # YOUR CODE: call anthropic_client.messages.create(
    # YOUR CODE:   model=ANTHROPIC_FALLBACK_MODEL,
    # YOUR CODE:   max_tokens=1024,
    # YOUR CODE:   tools=[FALLBACK_TOOL],
    # YOUR CODE:   tool_choice={"type": "tool", "name": "submit_invoice"},
    # YOUR CODE:   system=FALLBACK_SYSTEM,
    # YOUR CODE:   messages=[{"role": "user", "content": pdf_text}],
    # YOUR CODE: )
    # YOUR CODE: walk msg.content for a ToolUseBlock; grab block.input as a dict.
    # YOUR CODE: Validate it with Invoice.model_validate(...).
    # YOUR CODE: Set trace["used_fallback"] = True
    # YOUR CODE: Set trace["fallback_model"] = ANTHROPIC_FALLBACK_MODEL
    # YOUR CODE: On success: return (validated_instance, trace).
    # YOUR CODE: On ValidationError or no tool call: return (None, trace).

    return (None, trace)


# Pick a PDF known to fail GPT-4o-mini 3x. Scrambled-items invoices are the
# strongest trigger because the strict schema cannot fix mis-zipped columns
# even with error feedback.
_FB_TEST_INVOICE_NO = next(
    invno for invno, info in LABELS.items()
    if info.get("malformed") and info.get("kind") == "scrambled_items"
)
_FB_TEST_PATH = os.path.join(INVOICES_DIR, f"{_FB_TEST_INVOICE_NO}.pdf")
_FB_TEST_TEXT = extract_pdf_text(_FB_TEST_PATH)

print(f"Running fallback test on {_FB_TEST_INVOICE_NO} ...")
FB_INSTANCE, FB_TRACE = extract_with_fallback(_FB_TEST_TEXT)
print(f"OpenAI attempts: {FB_TRACE['attempts']}")
print(f"OpenAI status: {FB_TRACE['openai_status']}")
print(f"Used fallback: {FB_TRACE['used_fallback']}")
print(f"Fallback model: {FB_TRACE['fallback_model']}")
if FB_INSTANCE is not None:
    print(f"Final extraction: invoice_number={FB_INSTANCE.invoice_number!r} "
          f"vendor={FB_INSTANCE.vendor_name!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: the fallback engaged after 3 OpenAI attempts AND the final
# result came from Claude. We read the trace dict; we do not call the
# providers a second time.


def checkpoint_3(session):
    try:
        trace = FB_TRACE
        instance = FB_INSTANCE
    except NameError:
        return CheckpointResult(
            status="fail",
            error_reason="FB_TRACE / FB_INSTANCE are not defined; Task 3 cell did not run.",
        )

    if trace.get("attempts", 0) < 3:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"OpenAI fired only {trace.get('attempts', 0)} time(s); the fallback "
                "trigger is 3 consecutive failures. Confirm `extract_with_retry` is "
                "called with `max_retries=2` (total 3 attempts) and that retries are "
                "actually firing."
            ),
        )

    if trace.get("openai_status") != "exhausted":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"OpenAI status is {trace.get('openai_status')!r}; the fallback test "
                "PDF should exhaust GPT-4o-mini's retries. If OpenAI recovered, the "
                "test PDF picked was too easy; re-run the cell to pick another "
                "scrambled-items invoice from LABELS."
            ),
        )

    if not trace.get("used_fallback"):
        return CheckpointResult(
            status="fail",
            error_reason=(
                "trace['used_fallback'] is False. The fallback branch never ran. "
                "Confirm the Claude Sonnet call is inside `extract_with_fallback` and "
                "trace['used_fallback'] is set to True on the fallback path."
            ),
        )

    if trace.get("fallback_model") != ANTHROPIC_FALLBACK_MODEL:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"trace['fallback_model'] is {trace.get('fallback_model')!r}; expected "
                f"{ANTHROPIC_FALLBACK_MODEL!r}."
            ),
        )

    if instance is None:
        return CheckpointResult(
            status="fail",
            error_reason=(
                "Final extraction is None; Claude's tool_use output did not validate "
                "against the Pydantic schema. Check that `block.input` is being passed "
                "into `Invoice.model_validate(...)` and that the tool's input_schema "
                "is `Invoice.model_json_schema()`."
            ),
        )

    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Anthropic's tool-use response is a list of content blocks. Iterate them looking for a block whose `type == "tool_use"`. Its `.input` attribute is the dict the model passed to your tool, ready to feed into `Invoice.model_validate(...)`.

</details>

<details><summary>Hint 2 (direction)</summary>

The `tool_choice={"type": "tool", "name": "submit_invoice"}` forces Claude to call that specific tool every time. The forced-tool path makes the model's response a single tool_use block with the structured data inside. Wrap `Invoice.model_validate(block.input)` in `try / except ValidationError` so a malformed Claude response surfaces as a clean None return rather than an exception.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working fallback body:

```python
msg = anthropic_client.messages.create(
    model=ANTHROPIC_FALLBACK_MODEL,
    max_tokens=1024,
    tools=[FALLBACK_TOOL],
    tool_choice={"type": "tool", "name": "submit_invoice"},
    system=FALLBACK_SYSTEM,
    messages=[{"role": "user", "content": pdf_text}],
)
for block in msg.content:
    if getattr(block, "type", None) == "tool_use":
        try:
            validated = Invoice.model_validate(block.input)
        except ValidationError:
            return (None, trace)
        trace["used_fallback"] = True
        trace["fallback_model"] = ANTHROPIC_FALLBACK_MODEL
        return (validated, trace)
return (None, trace)
```

</details>

**Wallet check.** Three GPT-4o-mini calls plus one Claude Sonnet 4.6 call on a single PDF. Sonnet is roughly 20x the per-token cost of mini, but the fallback ran on one invoice only. Spent on this task: about $0.03. Cumulative session: roughly $0.05.

## Task 4: End-to-end batch across all 500 invoices, write CSV, hit 99%

Run `extract_with_fallback` over every PDF in `LABELS`. For each invoice, write one row to `/content/extraction_results.csv` with:

- `invoice_number`
- `status` (`ok` or `fail`)
- `openai_attempts`
- `used_fallback` (boolean)
- `fallback_model` (or blank)
- `vendor_name` (extracted value or blank on fail)

Print a progress line every 100 invoices with the running success count, retry count, and fallback count. The 100-line cadence matches the brief's voice: "100 done. 7 retries fired. 1 fallback. Spent: ~$0.30."

Checkpoint 4 reads the CSV back and confirms the success rate clears 99%.

In [ ]:
# NBVAL_SKIP
# Task 4: process all 500 invoices end-to-end and write the trace CSV.

CSV_COLUMNS = [
    "invoice_number",
    "status",
    "openai_attempts",
    "used_fallback",
    "fallback_model",
    "vendor_name",
]


def _write_results_row(writer, invoice_no, instance, trace, status):
    writer.writerow({
        "invoice_number": invoice_no,
        "status": status,
        "openai_attempts": trace.get("attempts", 0),
        "used_fallback": trace.get("used_fallback", False),
        "fallback_model": trace.get("fallback_model") or "",
        "vendor_name": instance.vendor_name if instance is not None else "",
    })


# YOUR CODE: open RESULTS_CSV for writing, csv.DictWriter with CSV_COLUMNS.
# YOUR CODE: writer.writeheader().
# YOUR CODE: for each invoice_no in LABELS.keys():
# YOUR CODE:   pdf_path = INVOICES_DIR/<invoice_no>.pdf
# YOUR CODE:   text = extract_pdf_text(pdf_path)
# YOUR CODE:   instance, trace = extract_with_fallback(text)
# YOUR CODE:   status = "ok" if instance is not None else "fail"
# YOUR CODE:   _write_results_row(writer, invoice_no, instance, trace, status)
# YOUR CODE:   bump counters: success_count, retry_count (sum of retries fired),
# YOUR CODE:                   fallback_count
# YOUR CODE:   every 100 invoices, print a progress line like the brief example.
# YOUR CODE: after the loop, print a final summary line.

success_count = 0
retry_count = 0
fallback_count = 0

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: the CSV exists, has 500 rows, and the success rate is >= 99%.


def checkpoint_4(session):
    if not os.path.exists(RESULTS_CSV):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{RESULTS_CSV} does not exist. Task 4 either failed or never ran. "
                "Confirm the writer wrote with `with open(...) as f:` so the buffer flushed."
            ),
        )

    rows = []
    try:
        with open(RESULTS_CSV) as f:
            reader = csv.DictReader(f)
            for row in reader:
                rows.append(row)
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"Could not read {RESULTS_CSV}: {exc!r}",
        )

    if len(rows) != TOTAL_INVOICES:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{RESULTS_CSV} has {len(rows)} rows; expected {TOTAL_INVOICES}. "
                "Confirm the Task 4 loop iterates every key in LABELS."
            ),
        )

    ok = sum(1 for r in rows if r.get("status") == "ok")
    rate = ok / TOTAL_INVOICES
    if rate < 0.99:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Success rate {rate:.3f} below 0.99 threshold "
                f"({ok}/{TOTAL_INVOICES} succeeded). "
                "Common cause: the PDF text extractor is dropping multi-line vendor names. "
                "Confirm `extract_pdf_text` uses PyMuPDF (`fitz`) rather than pdfplumber's "
                "default extractor, and that the retry prompt carries the validator error."
            ),
        )

    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The number under 99% means some invoices made it past the OpenAI retry but the fallback could not save them either, OR the extraction is succeeding but the row is being written wrong. Print one failing row to see what shape it has.

</details>

<details><summary>Hint 2 (direction)</summary>

PDF text extraction is the most common culprit. If `extract_pdf_text` returns "Vendor: Northeastern Regional" with the second line "Manufacturing Group" appearing as a totally separate text block somewhere else on the page, no amount of retry feedback fixes the vendor field. PyMuPDF (`fitz`) holds layout-adjacent lines together; pdfplumber's default extractor sometimes does not.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working Task 4 body:

```python
success_count = 0
retry_count = 0
fallback_count = 0

with open(RESULTS_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
    writer.writeheader()

    for i, invoice_no in enumerate(LABELS.keys(), start=1):
        pdf_path = os.path.join(INVOICES_DIR, f"{invoice_no}.pdf")
        text = extract_pdf_text(pdf_path)
        instance, trace = extract_with_fallback(text)
        status = "ok" if instance is not None else "fail"
        _write_results_row(writer, invoice_no, instance, trace, status)
        if status == "ok":
            success_count += 1
        retry_count += len(trace.get("retries", []))
        if trace.get("used_fallback"):
            fallback_count += 1
        if i % 100 == 0:
            spent = i * 0.0015  # rough $1.50 / 500 invoices estimate
            print(f"{i} done. {retry_count} retries fired. "
                  f"{fallback_count} fallback(s). Spent: ~${spent:.2f}.")

print(f"Final: {success_count}/{TOTAL_INVOICES} succeeded "
      f"({success_count / TOTAL_INVOICES * 100:.1f}%). "
      f"Retries fired: {retry_count}. Fallbacks: {fallback_count}.")
```

</details>

**Wallet check.** Five hundred PDFs through GPT-4o-mini, roughly 40 retries, and 5 to 15 fallbacks to Claude Sonnet 4.6. Total token spend lands in the $1.30 to $1.80 range for the batch. Plus the $0.05 from earlier tasks: cumulative session about $1.50 to $1.85.

## Cleanup

Empty and delete the S3 bucket, remove the local results CSV, and remove the local invoices directory. The verification scan re-queries S3 to confirm the bucket is gone and checks the filesystem for the local artifacts. Any orphan triggers a dirty exit.

In [ ]:
# NBVAL_SKIP
# Cleanup. Empty + delete the S3 bucket, drop the local results CSV, drop the
# invoices directory. Verification re-queries S3 and the filesystem.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $1.50 to $2.00.** Five hundred invoices extracted with strict Pydantic validation, error-feedback retries, and Claude Sonnet 4.6 fallback. GPT-4o-mini did the bulk of the work; Claude stepped in on the few cases where strict mode could not recover even with error feedback. The CSV trace is the artifact: every invoice has a row, every retry is countable, every fallback is logged. S3 bucket dropped. Local files gone.

500 invoices processed. PDFs gone. Results CSV deleted.

## Reflection

These are not graded. They are for you.

1. The pipeline hits 99% on the test set. The production traffic has a 12% rate of malformed PDFs your test set does not cover (handwriting on scanned faxes, three-column layouts, foreign-language vendor names). What is the most defensible quality gate you would add before running this against production traffic, and why? Walk through the trade-off between adding a third model tier, adding a human-review queue for low-confidence extractions, and gating writes to the downstream database on a confidence score.

2. The fallback to Claude Sonnet doubles the per-invoice token cost when it fires. The team is considering a "Sonnet-first, mini-as-validator" approach to lift the success rate floor. Beyond cost, what is one risk of that strategy? Think about which model carries the schema-binding contract and what happens when the cheaper model disagrees with the more expensive one.

## Exam-Style Practice

**Question 1 (structured output failure mode):**

A team's GPT-4o-mini pipeline with strict structured output is rejecting 8% of invoices. They want 99% success without changing models. Which intervention is highest leverage?

A. Switch to GPT-4o (full) and accept the 5x cost.
B. Add a retry loop where the next prompt includes the validator's `ValidationError.errors()` text.
C. Mark every schema field `Optional[...]` so strict mode passes more often.
D. Pre-process every PDF through an OCR pass before sending to the LLM.

<details><summary>Show answer</summary>

**Correct: B.**

- B is correct: in-context retry with the validator's error text is the documented OpenAI pattern for lifting structured-output success rates. An 8% rejection rate typically drops below 1% with no model change because the model gets a precise, line-numbered description of what was wrong and can correct on the next attempt.
- A is wrong: GPT-4o (full) does not understand the schema any better than mini; the cost rises 5x for no meaningful gain on this failure mode.
- C is wrong: marking fields Optional defeats the purpose of strict mode. The downstream database still needs every field; the failure just moves from validation to dirty data.
- D is wrong: the failure is in interpretation, not text extraction. OCR is irrelevant when the PDFs already contain native text.

</details>

**Question 2 (fallback routing trade-off):**

The team has two LLM providers. The cheaper primary fails 8% of the time. The fallback is roughly 20x the per-token cost. A junior engineer proposes routing every request to both models in parallel and picking the first valid response. What is the strongest objection?

A. Parallel calls double the latency.
B. Parallel calls double the token cost for the 92% of requests where the primary would have succeeded alone.
C. The two models cannot share a tool-use schema.
D. The fallback model returns lower-quality data.

<details><summary>Show answer</summary>

**Correct: B.**

- B is correct: the parallel-first design pays for the fallback on every request, including the 92% the primary would have handled solo. Sequential-with-fallback only pays the fallback price on the 8% that need it. For a 20x cost ratio that is the difference between a $1.50 batch and roughly $30.
- A is wrong: parallel calls typically lower latency because the slower call does not block; latency is not the strongest objection.
- C is wrong: tool-use input_schema is just JSON Schema; both OpenAI structured outputs and Anthropic tool-use accept the same shape.
- D is wrong: the fallback in this lab is more expensive precisely because it is more capable on this failure mode; the data quality argument cuts the other direction.

</details>

**Question 3 (schema contract change):**

The product team adds a new required field `tax_amount: float` to the invoice schema. The deployed pipeline still serves traffic. What is the safest sequencing for the schema change?

A. Update the Pydantic schema in code and redeploy. Old extractions without `tax_amount` will fail validation, which is fine because the new ones will be correct.
B. Add `tax_amount` as `Optional[float]` first, redeploy, backfill the missing field on old records via a one-off batch, then promote it to required in a second deploy.
C. Update the OpenAI prompt to request `tax_amount` but leave the Pydantic schema unchanged; validate the new field downstream.
D. Roll out the schema change to the fallback model first because it is more reliable.

<details><summary>Show answer</summary>

**Correct: B.**

- B is correct: the two-phase migration (`Optional` first, backfill, then required) is the standard pattern for adding a required field without breaking in-flight extractions. The intermediate `Optional[float]` state is what makes it safe.
- A is wrong: flipping a required field on a live pipeline guarantees the next 8% (or more) of valid invoices fail validation just because they have no `tax_amount` yet. The retry loop cannot fix data that does not exist.
- C is wrong: bypassing the schema for one field defeats the entire point of strict structured output. Drift between prompt and schema is exactly what this pipeline exists to prevent.
- D is wrong: the fallback's reliability is irrelevant to schema migration ordering; both models bind to the same schema.

</details>